# Data Cleaning

Clean missing values, types, duplicates, and invalid measurements.

In [1]:
import pandas as pd
import numpy as np
# Load a raw dataset here.

In [2]:
df = pd.read_csv("../data/raw/city_day.csv")

df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [3]:
df["Date"] = pd.to_datetime(df["Date"])


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   City        29531 non-null  str           
 1   Date        29531 non-null  datetime64[us]
 2   PM2.5       24933 non-null  float64       
 3   PM10        18391 non-null  float64       
 4   NO          25949 non-null  float64       
 5   NO2         25946 non-null  float64       
 6   NOx         25346 non-null  float64       
 7   NH3         19203 non-null  float64       
 8   CO          27472 non-null  float64       
 9   SO2         25677 non-null  float64       
 10  O3          25509 non-null  float64       
 11  Benzene     23908 non-null  float64       
 12  Toluene     21490 non-null  float64       
 13  Xylene      11422 non-null  float64       
 14  AQI         24850 non-null  float64       
 15  AQI_Bucket  24850 non-null  str           
dtypes: datetime64[us](1), float64(13)

In [5]:
df = df.sort_values(["City" , "Date"]).reset_index(drop = True)

In [6]:
df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.duplicated(subset = ["City" , "Date"]).sum()

np.int64(0)

In [9]:
df = df.drop(columns = ["Xylene" , "Toluene" , "Benzene"])

In [10]:
df.shape

(29531, 13)

In [11]:
df.columns

Index(['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2',
       'O3', 'AQI', 'AQI_Bucket'],
      dtype='str')

In [12]:
num_cols = df.select_dtypes(include = np.number).columns

(df[num_cols] < 0).sum()

PM2.5    0
PM10     0
NO       0
NO2      0
NOx      0
NH3      0
CO       0
SO2      0
O3       0
AQI      0
dtype: int64

In [13]:
city_coverage = df.groupby("City").agg(

    start_date = ("Date" , "min"),
    end_date = ("Date" , "max") , 
    records = ("Date" , "count")
)

city_coverage

,start_date,end_date,records
City,,,
Ahmedabad,2015-01-01,2020-07-01,2009
Aizawl,2020-03-11,2020-07-01,113
Amaravati,2017-11-24,2020-07-01,951
Amritsar,2017-02-27,2020-07-01,1221
Bengaluru,2015-01-01,2020-07-01,2009
Bhopal,2019-09-17,2020-07-01,289
Brajrajnagar,2017-12-07,2020-07-01,938
Chandigarh,2019-09-02,2020-07-01,304
Chennai,2015-01-01,2020-07-01,2009


In [14]:
core_cols = ["PM2.5" , "PM10" , "NO2", "CO" , "SO2" , "O3" , "AQI"]

city_missing = (
    
    df.groupby("City")[core_cols]
    .apply(lambda x : x.isnull().mean() * 100)
    .round(2)
)

city_missing

,PM2.5,PM10,NO2,CO,SO2,O3,AQI
City,,,,,,,
Ahmedabad,31.26,79.74,30.26,30.31,31.86,34.10,33.60
Aizawl,1.77,0.88,0.00,0.00,0.00,7.96,1.77
Amaravati,6.20,5.78,5.78,10.20,6.83,5.78,11.57
Amritsar,10.48,5.16,3.60,6.22,13.76,9.42,7.78
Bengaluru,7.27,17.92,0.30,0.55,0.30,7.17,4.93
Bhopal,3.11,3.11,3.11,3.11,3.11,3.11,3.81
Brajrajnagar,19.72,18.44,24.95,15.99,20.58,21.64,23.99
Chandigarh,4.93,0.00,1.32,0.00,0.00,0.00,1.64
Chennai,5.82,84.97,1.79,1.24,1.79,2.44,6.22


In [15]:
ml_features = ["PM2.5" , "NO2" , "CO" , "SO2" , "O3"]

quality = df.groupby("City").agg(
    
    records = ("Date" , "count") , 
    start_date = ("Date" , "min"),
    end_date = ("Date" , "max") , 
    aqi_missing = ("AQI" , lambda x : x.isna().mean() *100)
)

feature_missing = (
    
    df.groupby("City")[ml_features]
    .apply(lambda x : x.isna().mean().mean() * 100)
    .rename("feature_missing")
)

quality = quality.join(feature_missing)

quality = quality.round( {
    "aqi_missing" : 2,
    "feature_missing" : 2
})

quality.sort_values(["records"  , "aqi_missing"] , ascending = [False , True])

,records,start_date,end_date,aqi_missing,feature_missing
City,,,,,
Delhi,2009,2015-01-01,2020-07-01,0.50,1.97
Bengaluru,2009,2015-01-01,2020-07-01,4.93,3.12
Lucknow,2009,2015-01-01,2020-07-01,5.77,2.44
Chennai,2009,2015-01-01,2020-07-01,6.22,2.62
Ahmedabad,2009,2015-01-01,2020-07-01,33.60,31.56
Mumbai,2009,2015-01-01,2020-07-01,61.42,49.13
Hyderabad,2006,2015-01-04,2020-07-01,6.28,1.99
Patna,1858,2015-06-01,2020-07-01,21.47,12.48
Gurugram,1679,2015-11-27,2020-07-01,13.46,8.22


In [16]:
eligible_cities = quality[
    (quality["records"] >= 500) &
    (quality["aqi_missing"] <= 20) &
    (quality["feature_missing"] <= 25)
].index.tolist()

eligible_cities

['Amaravati',
 'Amritsar',
 'Bengaluru',
 'Chennai',
 'Delhi',
 'Gurugram',
 'Guwahati',
 'Hyderabad',
 'Jaipur',
 'Kolkata',
 'Lucknow',
 'Thiruvananthapuram',
 'Visakhapatnam']

In [17]:
df.to_csv(
    "../data/processed/air_quality_clean.csv",
    index=False
)

#### ML candidate dataset

In [18]:
ml_df = df[df["City"].isin(eligible_cities)].copy()

ml_df.shape

(18897, 13)

In [19]:
ml_df["City"].value_counts()

City
Bengaluru             2009
Chennai               2009
Delhi                 2009
Lucknow               2009
Hyderabad             2006
Gurugram              1679
Visakhapatnam         1462
Amritsar              1221
Jaipur                1114
Thiruvananthapuram    1112
Amaravati              951
Kolkata                814
Guwahati               502
Name: count, dtype: int64

In [20]:
def max_missing_streak(s):
    m = s.isna()
    groups = (m != m.shift()).cumsum()
    streaks = m.groupby(groups).sum()
    return int(streaks.max()) if len(streaks) else 0

In [21]:
check_cols = ["PM2.5", "NO2", "CO", "SO2", "O3", "AQI"]

streak_report = pd.DataFrame()

for col in check_cols:
    streak_report[col] = (
        ml_df.groupby("City")[col]
        .apply(max_missing_streak)
    )

streak_report

,PM2.5,NO2,CO,SO2,O3,AQI
City,,,,,,
Amaravati,37,37,37,37,37,38
Amritsar,60,9,15,51,50,18
Bengaluru,78,2,2,2,23,79
Chennai,81,17,11,17,17,82
Delhi,2,2,0,81,81,4
Gurugram,56,52,45,30,37,57
Guwahati,1,1,1,1,1,2
Hyderabad,86,19,1,19,19,86
Jaipur,5,5,5,5,5,6


In [22]:
def date_gap_summary(g):
    g = g.sort_values("Date")
    gaps = g["Date"].diff().dt.days

    return pd.Series({
        "max_gap_days": gaps.max(),
        "missing_calendar_days": (
            (g["Date"].max() - g["Date"].min()).days + 1 - len(g)
        )
    })

date_gaps = (
    ml_df.groupby("City")
    .apply(date_gap_summary, include_groups=False) # type: ignore
) # type: ignore

date_gaps

,max_gap_days,missing_calendar_days
City,,
Amaravati,1.0,0.0
Amritsar,1.0,0.0
Bengaluru,1.0,0.0
Chennai,1.0,0.0
Delhi,1.0,0.0
Gurugram,1.0,0.0
Guwahati,1.0,0.0
Hyderabad,1.0,0.0
Jaipur,1.0,0.0


In [23]:
ml_df = ml_df.sort_values(["City", "Date"]).reset_index(drop=True)

ml_df.to_csv(
    "../data/processed/ml_candidate.csv",
    index=False
)